# Real-Time MEV Transaction Classification

In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
import requests
from datetime import datetime, timedelta
import pyarrow.parquet as pq
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## 1. Load Pre-Downloaded Mempool Data

In [2]:
DATA_DIR = Path("data/mempool")

def get_downloaded_files():
    parquet_files = sorted(DATA_DIR.glob("*.parquet"))
    
    if not parquet_files:
        logger.error(f"No parquet files found in {DATA_DIR}")
        logger.info("Run: python download_mempool_historical.py")
        return []
    
    logger.info(f"Found {len(parquet_files)} parquet files:")
    for f in parquet_files:
        size_mb = f.stat().st_size / 1e6
        logger.info(f"  {f.name} ({size_mb:.1f} MB)")
    
    return parquet_files

downloaded_files = get_downloaded_files()

if downloaded_files:
    date_range = f"{downloaded_files[0].stem} to {downloaded_files[-1].stem}"
    logger.info(f"Date range: {date_range}")
    logger.info(f"Ready to load {len(downloaded_files)} files")

2025-11-08 19:09:29,956 - INFO - Found 8 parquet files:
2025-11-08 19:09:29,956 - INFO -   2025-10-31.parquet (4772.9 MB)
2025-11-08 19:09:29,957 - INFO -   2025-11-01.parquet (4750.3 MB)
2025-11-08 19:09:29,957 - INFO -   2025-11-02.parquet (4731.8 MB)
2025-11-08 19:09:29,957 - INFO -   2025-11-03.parquet (5190.7 MB)
2025-11-08 19:09:29,958 - INFO -   2025-11-04.parquet (5785.5 MB)
2025-11-08 19:09:29,958 - INFO -   2025-11-05.parquet (4620.6 MB)
2025-11-08 19:09:29,958 - INFO -   2025-11-06.parquet (4676.1 MB)
2025-11-08 19:09:29,958 - INFO -   2025-11-07.parquet (5632.1 MB)
2025-11-08 19:09:29,958 - INFO - Date range: 2025-10-31 to 2025-11-07
2025-11-08 19:09:29,958 - INFO - Ready to load 8 files
2025-11-08 19:09:29,956 - INFO -   2025-10-31.parquet (4772.9 MB)
2025-11-08 19:09:29,957 - INFO -   2025-11-01.parquet (4750.3 MB)
2025-11-08 19:09:29,957 - INFO -   2025-11-02.parquet (4731.8 MB)
2025-11-08 19:09:29,957 - INFO -   2025-11-03.parquet (5190.7 MB)
2025-11-08 19:09:29,958 - I

## 2. Load and Parse Mempool Data

In [3]:
def load_parquet_file(filepath):
    logger.info(f"Loading: {filepath.name}")
    
    df = pq.read_table(filepath).to_pandas()
    
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['includedBlockTimestamp'] = pd.to_datetime(df['includedBlockTimestamp'])
    
    df['gasPrice'] = pd.to_numeric(df['gasPrice'], errors='coerce')
    df['gas'] = pd.to_numeric(df['gas'], errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['nonce'] = pd.to_numeric(df['nonce'], errors='coerce')
    
    logger.info(f"Loaded {len(df):,} transactions")
    return df

def load_all_mempool_data(files):
    all_data = []
    
    for filepath in files:
        df = load_parquet_file(filepath)
        all_data.append(df)
    
    combined = pd.concat(all_data, ignore_index=True)
    combined = combined.sort_values('timestamp').reset_index(drop=True)
    
    logger.info(f"Total transactions loaded: {len(combined):,}")
    return combined

mempool_df = load_all_mempool_data(downloaded_files)
logger.info(f"Date range: {mempool_df['timestamp'].min()} to {mempool_df['timestamp'].max()}")
logger.info(f"Columns: {list(mempool_df.columns)}")

2025-11-08 19:09:29,963 - INFO - Loading: 2025-10-31.parquet
2025-11-08 19:09:33,540 - INFO - Loaded 1,114,285 transactions
2025-11-08 19:09:33,540 - INFO - Loading: 2025-11-01.parquet
2025-11-08 19:09:33,540 - INFO - Loaded 1,114,285 transactions
2025-11-08 19:09:33,540 - INFO - Loading: 2025-11-01.parquet
2025-11-08 19:09:38,308 - INFO - Loaded 945,695 transactions
2025-11-08 19:09:38,308 - INFO - Loading: 2025-11-02.parquet
2025-11-08 19:09:38,308 - INFO - Loaded 945,695 transactions
2025-11-08 19:09:38,308 - INFO - Loading: 2025-11-02.parquet
2025-11-08 19:09:44,980 - INFO - Loaded 1,048,641 transactions
2025-11-08 19:09:44,981 - INFO - Loading: 2025-11-03.parquet
2025-11-08 19:09:44,980 - INFO - Loaded 1,048,641 transactions
2025-11-08 19:09:44,981 - INFO - Loading: 2025-11-03.parquet
2025-11-08 19:09:52,574 - INFO - Loaded 1,125,550 transactions
2025-11-08 19:09:52,576 - INFO - Loading: 2025-11-04.parquet
2025-11-08 19:09:52,574 - INFO - Loaded 1,125,550 transactions
2025-11-08 1

## 3. FlashBoys Labeling: Post-Hoc Auction Detection

In [ ]:
class FlashBoysLabeler:
    def __init__(self, time_window=2.0, min_auction_size=2, min_price_escalation=1.1):
        self.time_window = time_window
        self.min_auction_size = min_auction_size
        self.min_price_escalation = min_price_escalation
    
    def detect_gas_auctions(self, df, max_transactions=None):
        logger.info("Running FlashBoys auction detection")
        
        if max_transactions and len(df) > max_transactions:
            logger.info(f"Sampling {max_transactions:,} from {len(df):,} transactions")
            df = df.sample(n=max_transactions, random_state=42).sort_values('timestamp').reset_index(drop=True)
        
        df = df.sort_values('timestamp').reset_index(drop=True)
        df['is_mev_auction'] = 0
        df['auction_id'] = -1
        df['to_filled'] = df['to'].fillna('0x0')
        df['timestamp_sec'] = df['timestamp'].astype(np.int64) / 1e9
        
        auction_id = 0
        processed = set()
        idx = 0
        
        pbar = tqdm(total=len(df), desc="Detecting auctions")
        
        while idx < len(df):
            pbar.update(1)
            
            if idx in processed:
                idx += 1
                continue
            
            target_contract = df.loc[idx, 'to_filled']
            if target_contract == '0x0':
                idx += 1
                continue
            
            base_time = df.loc[idx, 'timestamp_sec']
            max_time = base_time + self.time_window
            
            window_end = min(idx + 100, len(df))
            window_slice = df.iloc[idx:window_end]
            
            mask = (
                (window_slice['to_filled'] == target_contract) &
                (window_slice['timestamp_sec'] <= max_time) &
                (~window_slice.index.isin(processed))
            )
            
            candidates = window_slice[mask].index.tolist()
            
            if len(candidates) < self.min_auction_size:
                idx += 1
                continue
            
            gas_prices = df.loc[candidates, 'gasPrice'].values
            
            is_escalating = all(
                gas_prices[k + 1] > gas_prices[k] * self.min_price_escalation
                for k in range(len(gas_prices) - 1)
            )
            
            if not is_escalating:
                idx += 1
                continue
            
            senders = df.loc[candidates, 'from'].values
            nonces = df.loc[candidates, 'nonce'].values
            
            sender_nonce_pairs = list(zip(senders, nonces))
            has_replacements = len(sender_nonce_pairs) != len(set(sender_nonce_pairs))
            unique_senders = len(set(senders))
            
            if has_replacements or unique_senders >= 2:
                df.loc[candidates, 'is_mev_auction'] = 1
                df.loc[candidates, 'auction_id'] = auction_id
                processed.update(candidates)
                auction_id += 1
                idx = max(candidates) + 1
            else:
                idx += 1
        
        pbar.close()
        
        num_auctions = auction_id
        num_mev_txs = (df['is_mev_auction'] == 1).sum()
        mev_ratio = num_mev_txs / len(df) if len(df) > 0 else 0
        
        logger.info(f"Detected {num_auctions:,} gas auctions")
        logger.info(f"MEV transactions: {num_mev_txs:,} ({100*mev_ratio:.2f}%)")
        logger.info(f"Expected range: 10-15%")
        
        if mev_ratio > 0.20:
            logger.warning(f"MEV ratio {100*mev_ratio:.2f}% exceeds 20% threshold")
        
        return df

labeler = FlashBoysLabeler(time_window=2.0, min_auction_size=2, min_price_escalation=1.1)
mempool_df = labeler.detect_gas_auctions(mempool_df, max_transactions=1000000)

logger.info(f"Label distribution: {mempool_df['is_mev_auction'].value_counts().to_dict()}")


2025-11-08 19:11:00,009 - INFO - Running FlashBoys auction detection
2025-11-08 19:11:00,011 - INFO - Sampling 1,000,000 from 8,775,676 transactions
2025-11-08 19:11:00,011 - INFO - Sampling 1,000,000 from 8,775,676 transactions


## 4. Feature Engineering: Causal Feature Extraction

In [ ]:
class MempoolFeatureExtractor:
    def __init__(self, lookback_window=20):
        self.lookback_window = lookback_window
    
    def extract_features(self, df):
        logger.info("Extracting causal features")
        
        features = []
        labels = []
        
        for idx in tqdm(range(self.lookback_window, len(df)), desc="Feature extraction"):
            window_start = max(0, idx - self.lookback_window)
            window = df.iloc[window_start:idx]
            
            if len(window) < 5:
                continue
            
            current_gas = df.loc[idx, 'gasPrice']
            current_to = df.loc[idx, 'to']
            current_from = df.loc[idx, 'from']
            current_nonce = df.loc[idx, 'nonce']
            
            gas_prices = window['gasPrice'].values
            timestamps = window['timestamp'].values
            time_span = (timestamps[-1] - timestamps[0]) / np.timedelta64(1, 's') if len(timestamps) > 1 else 0
            
            feat = {
                'tx_index': idx,
                'gas_price': current_gas,
                'gas_limit': df.loc[idx, 'gas'],
                'tx_value': df.loc[idx, 'value'],
                'recent_gas_mean': np.mean(gas_prices),
                'recent_gas_std': np.std(gas_prices),
                'recent_gas_median': np.median(gas_prices),
                'recent_gas_max': np.max(gas_prices),
                'recent_gas_min': np.min(gas_prices),
                'gas_vs_mean': current_gas / (np.mean(gas_prices) + 1e-9),
                'gas_vs_median': current_gas / (np.median(gas_prices) + 1e-9),
                'gas_vs_max': current_gas / (np.max(gas_prices) + 1e-9),
                'tx_density': len(window) / (time_span + 1),
            }
            
            if pd.notna(current_to):
                same_target_recent = (window['to'] == current_to).sum()
                feat['same_target_count'] = same_target_recent
                feat['same_target_ratio'] = same_target_recent / len(window)
                
                target_txs = window[window['to'] == current_to]
                if len(target_txs) > 0:
                    feat['target_gas_max'] = target_txs['gasPrice'].max()
                    feat['gas_vs_target_max'] = current_gas / (target_txs['gasPrice'].max() + 1e-9)
                    feat['beating_target_max'] = 1 if current_gas > target_txs['gasPrice'].max() else 0
                else:
                    feat['target_gas_max'] = 0
                    feat['gas_vs_target_max'] = 1
                    feat['beating_target_max'] = 0
            else:
                feat['same_target_count'] = 0
                feat['same_target_ratio'] = 0
                feat['target_gas_max'] = 0
                feat['gas_vs_target_max'] = 1
                feat['beating_target_max'] = 0
            
            sender_prev_txs = window[window['from'] == current_from]
            if len(sender_prev_txs) > 0:
                feat['sender_recent_count'] = len(sender_prev_txs)
                feat['sender_has_same_nonce'] = 1 if (sender_prev_txs['nonce'] == current_nonce).any() else 0
                feat['sender_max_gas'] = sender_prev_txs['gasPrice'].max()
                feat['gas_vs_sender_max'] = current_gas / (sender_prev_txs['gasPrice'].max() + 1e-9)
            else:
                feat['sender_recent_count'] = 0
                feat['sender_has_same_nonce'] = 0
                feat['sender_max_gas'] = 0
                feat['gas_vs_sender_max'] = 1
            
            if len(gas_prices) >= 5:
                recent_5 = gas_prices[-5:]
                price_changes = np.diff(recent_5)
                feat['gas_momentum'] = np.mean(price_changes)
                feat['gas_acceleration'] = price_changes[-1] - price_changes[0] if len(price_changes) >= 2 else 0
                
                increasing = sum(1 for x in price_changes if x > 0)
                feat['price_increase_ratio'] = increasing / len(price_changes)
            else:
                feat['gas_momentum'] = 0
                feat['gas_acceleration'] = 0
                feat['price_increase_ratio'] = 0
            
            threshold_high = np.percentile(gas_prices, 90)
            feat['high_gas_count'] = (gas_prices > threshold_high).sum()
            feat['high_gas_ratio'] = feat['high_gas_count'] / len(gas_prices)
            feat['current_is_high_gas'] = 1 if current_gas > threshold_high else 0
            
            features.append(feat)
            labels.append(df.loc[idx, 'is_mev_auction'])
        
        feature_df = pd.DataFrame(features)
        feature_df['label'] = labels
        
        logger.info(f"Extracted {len(feature_df):,} feature vectors")
        logger.info(f"Feature columns: {len(feature_df.columns)}")
        
        return feature_df

extractor = MempoolFeatureExtractor(lookback_window=20)

sample_size = min(500000, len(mempool_df))
sample_df = mempool_df.iloc[:sample_size].copy()
logger.info(f"Processing sample of {len(sample_df):,} transactions")

feature_df = extractor.extract_features(sample_df)

logger.info(f"Feature shape: {feature_df.shape}")
logger.info(f"Label distribution: {feature_df['label'].value_counts().to_dict()}")


## 5. Train/Test Split: Temporal Validation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = [col for col in feature_df.columns if col not in ['label', 'tx_index']]

X = feature_df[feature_cols].fillna(0)
y = feature_df['label']

split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

logger.info(f"Train set: {len(X_train):,} samples")
logger.info(f"Test set: {len(X_test):,} samples")
logger.info(f"Train MEV ratio: {y_train.mean():.4f}")
logger.info(f"Test MEV ratio: {y_test.mean():.4f}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

logger.info("Feature scaling complete")

## 6. Train ML Model: LightGBM Classifier

In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

logger.info("Training LightGBM model...")

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum() if (y_train == 1).sum() > 0 else 1

model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model.fit(
    X_train_scaled, 
    y_train,
    eval_set=[(X_test_scaled, y_test)],
    callbacks=[lgb.log_evaluation(period=50)]
)

logger.info("Training complete")

## 7. Model Evaluation: Prediction Performance

In [ ]:
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

logger.info("\nClassification Report:")
logger.info(f"\n{classification_report(y_test, y_pred)}")

logger.info("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
logger.info(f"\n{cm}")

roc_auc = roc_auc_score(y_test, y_pred_proba)
logger.info(f"\nROC AUC Score: {roc_auc:.4f}")

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

logger.info("\nTop 10 Most Important Features:")
for idx, row in feature_importance.head(10).iterrows():
    logger.info(f"  {row['feature']}: {row['importance']:.4f}")

## 8. Model Evaluation: Detection Accuracy

In [ ]:
def analyze_detection_performance(feature_df, sample_df, predictions, probabilities, threshold=0.5):
    results = feature_df.copy()
    results['prediction'] = predictions
    results['probability'] = probabilities
    results['predicted_mev'] = (probabilities >= threshold).astype(int)
    results['actual_mev'] = results['label']
    
    auction_detection = []
    
    auction_groups = sample_df.iloc[results.index].groupby('auction_id')
    
    for auction_id, auction_df in auction_groups:
        if auction_id == -1:
            continue
        
        auction_indices = auction_df.index.tolist()
        
        if len(auction_indices) < 2:
            continue
        
        auction_start_time = sample_df.loc[auction_indices[0], 'timestamp']
        auction_end_time = sample_df.loc[auction_indices[-1], 'timestamp']
        auction_duration = (auction_end_time - auction_start_time).total_seconds()
        
        predicted_txs = results.loc[results.index.isin(auction_indices) & (results['predicted_mev'] == 1)]
        
        detection_rate = len(predicted_txs) / len(auction_indices)
        detected_auction = len(predicted_txs) > 0
        
        auction_detection.append({
            'auction_id': auction_id,
            'auction_size': len(auction_indices),
            'auction_duration_sec': auction_duration,
            'detected': detected_auction,
            'detection_rate': detection_rate,
            'txs_detected': len(predicted_txs)
        })
    
    detection_df = pd.DataFrame(auction_detection)
    
    if len(detection_df) > 0:
        logger.info("\nAuction Detection Performance:")
        logger.info(f"  Total auctions in test set: {len(detection_df):,}")
        logger.info(f"  Auctions detected (>=1 tx): {detection_df['detected'].sum():,} ({100*detection_df['detected'].mean():.1f}%)")
        logger.info(f"  Mean detection rate per auction: {100*detection_df['detection_rate'].mean():.1f}%")
        logger.info(f"  Mean auction duration: {detection_df['auction_duration_sec'].mean():.2f}s")
        logger.info(f"  Mean auction size: {detection_df['auction_size'].mean():.1f} txs")
        
        logger.info("\nTransaction-Level Metrics:")
        tp = ((results['actual_mev'] == 1) & (results['predicted_mev'] == 1)).sum()
        fp = ((results['actual_mev'] == 0) & (results['predicted_mev'] == 1)).sum()
        tn = ((results['actual_mev'] == 0) & (results['predicted_mev'] == 0)).sum()
        fn = ((results['actual_mev'] == 1) & (results['predicted_mev'] == 0)).sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        logger.info(f"  Precision: {precision:.3f}")
        logger.info(f"  Recall: {recall:.3f}")
        logger.info(f"  F1 Score: {f1:.3f}")
    else:
        logger.info("\nNo auctions found in test set")
    
    return detection_df

test_predictions = y_pred
test_probabilities = y_pred_proba
test_feature_df = feature_df.iloc[split_idx:]

detection_results = analyze_detection_performance(
    test_feature_df, 
    sample_df,
    test_predictions, 
    test_probabilities,
    threshold=0.5
)

## 9. Visualization: Model Performance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

top_features = feature_importance.head(15)
axes[0, 0].barh(top_features['feature'], top_features['importance'])
axes[0, 0].set_xlabel('Importance')
axes[0, 0].set_title('Top 15 Feature Importances')
axes[0, 0].invert_yaxis()

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[0, 1].plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve')
axes[0, 1].legend()

cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', ax=axes[1, 0])
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_title('Confusion Matrix (Normalized)')

if len(detection_results) > 0:
    axes[1, 1].hist(detection_results['auction_duration_sec'], bins=50, edgecolor='black')
    axes[1, 1].set_xlabel('Auction Duration (seconds)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Gas Auction Duration Distribution')
    axes[1, 1].axvline(detection_results['auction_duration_sec'].median(), 
                       color='red', linestyle='--', label=f'Median: {detection_results["auction_duration_sec"].median():.2f}s')
    axes[1, 1].legend()

plt.tight_layout()
plt.savefig('model_performance.png', dpi=300, bbox_inches='tight')
logger.info("Saved visualization: model_performance.png")
plt.show()

## 10. Save Model and Results

In [ ]:
import joblib

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "mev_detector_lightgbm.pkl"
scaler_path = MODEL_DIR / "feature_scaler.pkl"
feature_path = MODEL_DIR / "feature_columns.txt"

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)
with open(feature_path, 'w') as f:
    f.write('\n'.join(feature_cols))

logger.info(f"Model saved: {model_path}")
logger.info(f"Scaler saved: {scaler_path}")
logger.info(f"Features saved: {feature_path}")

results_path = MODEL_DIR / "evaluation_results.txt"
with open(results_path, 'w') as f:
    f.write("MEV Front-Running Detection Model Results\n")
    f.write("="*50 + "\n\n")
    f.write(f"Training samples: {len(X_train):,}\n")
    f.write(f"Test samples: {len(X_test):,}\n")
    f.write(f"ROC AUC: {roc_auc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_test, y_pred))
    f.write("\n\nTop 10 Features:\n")
    for idx, row in feature_importance.head(10).iterrows():
        f.write(f"  {row['feature']}: {row['importance']:.4f}\n")
    
    if len(detection_results) > 0:
        f.write("\n\nAuction Detection:\n")
        f.write(f"  Auctions detected: {detection_results['detected'].sum()}/{len(detection_results)}\n")
        f.write(f"  Detection rate: {100*detection_results['detected'].mean():.1f}%\n")
        f.write(f"  Mean auction duration: {detection_results['auction_duration_sec'].mean():.2f}s\n")

logger.info(f"Results saved: {results_path}")
logger.info("\nModel training complete")

## 11. Real-Time Inference Example

In [ ]:
class RealtimeMEVDetector:
    def __init__(self, model, scaler, feature_cols, lookback=20):
        self.model = model
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.lookback = lookback
        self.tx_buffer = []
    
    def add_transaction(self, tx):
        self.tx_buffer.append(tx)
        if len(self.tx_buffer) > self.lookback:
            self.tx_buffer.pop(0)
    
    def predict(self):
        if len(self.tx_buffer) < 2:
            return 0.0, False
        
        window = pd.DataFrame(self.tx_buffer)
        gas_prices = window['gasPrice'].values
        
        features = {
            'gas_price': self.tx_buffer[-1]['gasPrice'],
            'gas_limit': self.tx_buffer[-1]['gas'],
            'tx_value': self.tx_buffer[-1]['value'],
            'recent_gas_mean': np.mean(gas_prices),
            'recent_gas_std': np.std(gas_prices),
            'recent_gas_median': np.median(gas_prices),
            'recent_gas_max': np.max(gas_prices),
            'recent_gas_min': np.min(gas_prices),
            'gas_vs_mean': gas_prices[-1] / (np.mean(gas_prices) + 1e-9),
            'gas_vs_median': gas_prices[-1] / (np.median(gas_prices) + 1e-9),
            'gas_vs_max': gas_prices[-1] / (np.max(gas_prices) + 1e-9),
            'tx_density': len(window),
            'unique_senders': window['from'].nunique(),
            'unique_receivers': window['to'].nunique(),
        }
        
        if len(gas_prices) >= 3:
            recent_3 = gas_prices[-3:]
            price_changes = np.diff(recent_3)
            features['gas_momentum'] = np.mean(price_changes)
            features['gas_acceleration'] = price_changes[-1] - price_changes[0] if len(price_changes) >= 2 else 0
            increasing = sum(1 for x in price_changes if x > 0)
            features['price_increase_ratio'] = increasing / len(price_changes)
        else:
            features['gas_momentum'] = 0
            features['gas_acceleration'] = 0
            features['price_increase_ratio'] = 0
        
        threshold_high = np.percentile(gas_prices, 75)
        features['high_gas_count'] = (gas_prices > threshold_high).sum()
        features['high_gas_ratio'] = features['high_gas_count'] / len(gas_prices)
        
        sender_nonces = window.groupby('from')['nonce'].apply(list)
        features['sender_nonce_gaps'] = sum(
            sum(1 for i in range(len(nonces)-1) if nonces[i+1] != nonces[i] + 1)
            for nonces in sender_nonces if len(nonces) > 1
        )
        
        X = pd.DataFrame([features])[self.feature_cols].fillna(0)
        X_scaled = self.scaler.transform(X)
        
        probability = self.model.predict_proba(X_scaled)[0, 1]
        prediction = probability >= 0.5
        
        return probability, prediction

detector = RealtimeMEVDetector(model, scaler, feature_cols, lookback=20)

logger.info("\nSimulating real-time detection on test data...")
test_sample = sample_df.iloc[split_idx:split_idx+100]

for idx, row in test_sample.iterrows():
    tx = {
        'gasPrice': row['gasPrice'],
        'gas': row['gas'],
        'value': row['value'],
        'from': row['from'],
        'to': row['to'],
        'nonce': row['nonce']
    }
    
    detector.add_transaction(tx)
    prob, pred = detector.predict()
    
    if pred:
        logger.info(f"MEV Alert: Transaction {idx} - Probability: {prob:.2%}")

logger.info("\nReal-time detector ready for deployment")